In [1]:
# Imports
import numpy as np
import math
from collections import deque
from sklearn.datasets import load_iris
np.random.seed(42)

## 1. Load Iris Dataset
We use the classic Iris dataset via `sklearn.datasets.load_iris()`.

In [2]:
# Load Iris data (150 samples, 4 features, 3 classes)
iris = load_iris()
X_raw = iris.data.astype(float)
X_raw[:5]

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2]])

## 2. Preprocessing
We standardize features (z-score) so algorithms are scale-invariant.

In [4]:
def zscore_standardize(X):
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std[std == 0] = 1.0
    return (X - mean) / std, mean, std

X, mean, std = zscore_standardize(X_raw)
X.shape, X[:3]

((150, 4),
 array([[-0.90068117,  1.01900435, -1.34022653, -1.3154443 ],
        [-1.14301691, -0.13197948, -1.34022653, -1.3154443 ],
        [-1.38535265,  0.32841405, -1.39706395, -1.3154443 ]]))

## 3. Helper Functions
Distance computations and utility helpers.

In [3]:
def pairwise_distances(X):
    # Euclidean pairwise distance matrix
    # Efficient via (x - y)^2 = x^2 + y^2 - 2xy
    sq = np.sum(X**2, axis=1, keepdims=True)
    D2 = sq + sq.T - 2 * (X @ X.T)
    D2 = np.maximum(D2, 0.0)
    return np.sqrt(D2)

def neighbors_within_eps(D, i, eps):
    return np.where(D[i] <= eps)[0]

## 4. DBSCAN (from scratch)

In [5]:
def dbscan(X, eps=0.5, min_samples=5):
    n = X.shape[0]
    D = pairwise_distances(X)
    labels = np.full(n, -1, dtype=int)  # -1: noise
    visited = np.zeros(n, dtype=bool)
    cluster_id = 0

    for i in range(n):
        if visited[i]:
            continue
        visited[i] = True
        nbrs = neighbors_within_eps(D, i, eps)
        if nbrs.size < min_samples:
            labels[i] = -1  # noise
        else:
            # expand cluster
            labels[i] = cluster_id
            queue = deque(nbrs.tolist())
            while queue:
                j = queue.popleft()
                if not visited[j]:
                    visited[j] = True
                    j_nbrs = neighbors_within_eps(D, j, eps)
                    if j_nbrs.size >= min_samples:
                        for k in j_nbrs:
                            if labels[k] == -1:
                                labels[k] = cluster_id
                            if labels[k] == -1 or not visited[k]:
                                queue.append(k)
                if labels[j] == -1:
                    labels[j] = cluster_id
            cluster_id += 1
    return labels, cluster_id

db_labels, db_n_clusters = dbscan(X, eps=0.6, min_samples=6)
db_n_clusters, np.bincount(np.where(db_labels>=0, db_labels, 0))

(2, array([75, 75]))

## 5. K-Means (from scratch)

In [6]:
def kmeans(X, k, max_iters=100, tol=1e-4):
    n, d = X.shape
    # init: k random points
    idx = np.random.choice(n, k, replace=False)
    centers = X[idx].copy()
    labels = np.zeros(n, dtype=int)
    for it in range(max_iters):
        # assign step
        D = pairwise_distances(X)  # we'll compute to centers below more efficiently
        # compute distances to centers
        # dist(X_i, center_j)^2 = ||X_i||^2 + ||c_j||^2 - 2 X_i dot c_j
        x_sq = np.sum(X**2, axis=1, keepdims=True)
        c_sq = np.sum(centers**2, axis=1, keepdims=True).T
        d2 = x_sq + c_sq - 2 * (X @ centers.T)
        labels = np.argmin(d2, axis=1)
        # update step
        new_centers = np.zeros_like(centers)
        for j in range(k):
            pts = X[labels == j]
            if pts.size == 0:
                # reinitialize to a random point to avoid empty cluster
                new_centers[j] = X[np.random.randint(0, n)]
            else:
                new_centers[j] = pts.mean(axis=0)
        shift = np.linalg.norm(new_centers - centers)
        centers = new_centers
        if shift < tol:
            break
    return labels, centers

km_labels, km_centers = kmeans(X, k=3)
km_centers, np.bincount(km_labels)

(array([[-0.01139555, -0.87600831,  0.37707573,  0.31115341],
        [-1.01457897,  0.85326268, -1.30498732, -1.25489349],
        [ 1.16743407,  0.14530299,  1.00302557,  1.0300019 ]]),
 array([56, 50, 44]))

## 6. Comparison Metrics
We'll compute simple internal metrics: silhouette-like score approximation and cluster counts.

In [7]:
def average_intra_cluster_distance(X, labels):
    # average distance of each point to its cluster members
    D = pairwise_distances(X)
    clusters = np.unique(labels[labels >= 0])
    vals = []
    for c in clusters:
        idx = np.where(labels == c)[0]
        if idx.size <= 1:
            vals.append(0.0)
        else:
            sub = D[np.ix_(idx, idx)]
            vals.append(np.mean(sub))
    return np.mean(vals) if vals else float('inf')

def average_inter_cluster_center_distance(centers):
    if centers is None or len(centers) <= 1:
        return 0.0
    D = pairwise_distances(centers)
    # upper triangle mean excluding diagonal
    triu = D[np.triu_indices(D.shape[0], 1)]
    return triu.mean()

db_intra = average_intra_cluster_distance(X, db_labels)
km_intra = average_intra_cluster_distance(X, km_labels)
km_inter = average_inter_cluster_center_distance(km_centers)
print('DBSCAN: clusters =', len(np.unique(db_labels[db_labels>=0])), 'intra =', round(db_intra,3), 'noise =', np.sum(db_labels==-1))
print('K-Means: clusters =', len(np.unique(km_labels)), 'intra =', round(km_intra,3), 'inter =', round(km_inter,3))

DBSCAN: clusters = 2 intra = 1.139 noise = 29
K-Means: clusters = 3 intra = 1.207 inter = 2.95


## 7. Quick Visualization (optional numbers)
We won't use plotting libraries to keep dependencies minimal. We'll show sample points and centers numerically.

In [ ]:
# Show a few labeled samples for each method
def sample_points(X, labels, n=5):
    res = {}
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        res[int(c)] = X[idx[:n]]
    return res

print('DBSCAN sample points by label:', sample_points(X, db_labels))
print('K-Means sample points by label:', sample_points(X, km_labels))